# Clase 11 — Regresión logística y modelos lineales generalizados

**Curso:** Estadística Aplicada con Python · **Bloque 4:** Modelos y experimentos
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/11_regresion_logistica_glm.md`

## Qué vas a ver en esta clase

- Por qué la regresión lineal no basta para un resultado 0/1, y el modelo **logístico**: probabilidad → odds → log-odds y la función sigmoide.
- Ajustar con `statsmodels` (`smf.logit` y `sm.GLM` con familia Binomial) y **leer la salida**.
- Los coeficientes son **log-odds**; `exp(β)` es el **odds ratio**. Odds ratio con intervalo, odds ratio frente a riesgo relativo y variables categóricas (dummies).
- Probabilidades previstas y **efecto marginal medio**: el cambio en probabilidad depende del punto de partida.
- La **separación perfecta** (un aviso) y la familia de los **GLM**: lineal, logística y Poisson (esta última solo como idea).

Tres experimentos guiados: la recta frente a la logística en las previsiones, odds ratio frente a probabilidad y el efecto marginal.

## Requisitos

- Clase 10 (regresión lineal y lectura de un resultado de `statsmodels`), clases 6 y 7 (error estándar, intervalos, p-value).
- Librerías: `numpy`, `pandas`, `matplotlib`, `statsmodels` (`pip install numpy pandas matplotlib statsmodels`; ya vienen instaladas en Google Colab).

> **Cómo usar este notebook:** ejecuta las celdas en orden. Antes de cada experimento escribe qué esperas que ocurra. Los datos de abandono de clientes (*churn*) son sintéticos: un *ejemplo ilustrativo* generado con coeficientes verdaderos conocidos, para poder comparar la estimación con la verdad. La multicolinealidad, la selección de variables y la validación se ven en la clase 12.

---
## Celda 0 — Setup

Importamos las librerías, fijamos la **semilla** y definimos el generador de clientes sintéticos con sus **coeficientes verdaderos** (`VERDAD`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

# Coeficientes VERDADEROS con los que generamos los datos (solo los conocemos porque son sintéticos)
VERDAD = {"Intercept": -1.8, "antiguedad": -0.03, "gasto": 0.03, "datos_gb": 0.0, "tickets": 0.45, "plan_anual": -0.9}

def generar_clientes(n, rng):
    """Genera n clientes sintéticos (ejemplo ilustrativo) con abandono según VERDAD."""
    antiguedad = np.clip(rng.gamma(2.0, 14, n), 1, 72).round()
    gasto = np.clip(30 + 0.25 * antiguedad + rng.normal(0, 10, n), 10, None).round(1)
    datos_gb = np.clip(0.09 * gasto + rng.normal(0, 0.25, n), 0.1, None).round(2)   # la usaremos en la clase 12
    tickets = rng.poisson(np.exp(0.008 * (gasto - 40)))
    plan_anual = rng.binomial(1, 0.35, n)
    eta = (VERDAD["Intercept"] + VERDAD["antiguedad"] * antiguedad + VERDAD["gasto"] * gasto
           + VERDAD["tickets"] * tickets + VERDAD["plan_anual"] * plan_anual)
    abandona = rng.binomial(1, 1 / (1 + np.exp(-eta)))
    return pd.DataFrame({"abandona": abandona, "antiguedad": antiguedad, "gasto": gasto,
                         "datos_gb": datos_gb, "tickets": tickets, "plan_anual": plan_anual})

print("Librerías y funciones cargadas correctamente.")

---
## 1. Los datos: abandono de clientes (*churn*)

Generamos 2 000 clientes. La respuesta `abandona` vale 1 si el cliente se da de baja. Las explicativas son la antigüedad (meses), el gasto mensual (€), los tickets de soporte y el plan anual (0/1). El dataset incluye también `datos_gb`, que no usaremos hasta la clase 12.

In [ ]:
df = generar_clientes(2000, rng)

display(df.head())
print("Tasa de abandono:", round(df["abandona"].mean() * 100, 1), "%")
display(df.describe().round(2))

---
## 2. ¿Por qué no basta la regresión lineal?

Ajustamos una regresión lineal (clase 10) con `abandona` como respuesta y miramos las predicciones.

In [ ]:
ols = smf.ols('abandona ~ antiguedad + gasto + tickets + plan_anual', data=df).fit()
pred_ols = ols.predict(df)
print("Predicción mínima:", round(pred_ols.min(), 3), " máxima:", round(pred_ols.max(), 3))
print("% de clientes con 'probabilidad' fuera de [0, 1]:", round(((pred_ols < 0) | (pred_ols > 1)).mean() * 100, 1))

plt.hist(pred_ols, bins=40, edgecolor="white")
plt.axvline(0, color="red", linestyle="--")
plt.title("Predicciones de la regresión lineal sobre un resultado 0/1")
plt.xlabel("'probabilidad' prevista")
plt.show()

**¿Qué observas?** Hay predicciones negativas: una "probabilidad" menor que 0 no tiene sentido. Además la respuesta solo vale 0 o 1, así que los residuos no pueden ser normales.

### Experimento guiado 1: la recta y la logística al extrapolar

Repetimos el estudio **200 veces** con muestras de 500 clientes y ajustamos las dos versiones. Miramos la predicción para dos perfiles extremos: un cliente de riesgo (6 meses, 50 €, **8 tickets**, sin plan anual) y uno muy estable (72 meses, 20 €, 0 tickets, con plan anual). **Antes de ejecutar, predice:** ¿en cuántas muestras la recta dará una "probabilidad" fuera de [0, 1]? ¿Y la logística?

In [ ]:
perfiles_ext = pd.DataFrame({"antiguedad": [6, 72], "gasto": [50, 20], "tickets": [8, 0], "plan_anual": [0, 1]},
                            index=["riesgo", "estable"])
rng_e1 = np.random.default_rng(123)
prev_ols, prev_logit = [], []
for _ in range(200):
    d = generar_clientes(500, rng_e1)
    prev_ols.append(smf.ols('abandona ~ antiguedad + gasto + tickets + plan_anual', data=d).fit().predict(perfiles_ext).values)
    prev_logit.append(smf.logit('abandona ~ antiguedad + gasto + tickets + plan_anual', data=d).fit(disp=0).predict(perfiles_ext).values)
prev_ols, prev_logit = np.array(prev_ols), np.array(prev_logit)

resumen = pd.DataFrame({
    "recta: % fuera de [0, 1]": ((prev_ols < 0) | (prev_ols > 1)).mean(axis=0) * 100,
    "recta: media": prev_ols.mean(axis=0),
    "logística: mínimo": prev_logit.min(axis=0),
    "logística: máximo": prev_logit.max(axis=0)}, index=perfiles_ext.index)
display(resumen.round(3))

fig, ejes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for j, ax in enumerate(ejes):
    ax.hist(prev_ols[:, j], bins=25, alpha=0.6, label="recta")
    ax.hist(prev_logit[:, j], bins=25, alpha=0.6, label="logística")
    ax.axvline(0, color="red", linestyle="--"); ax.axvline(1, color="red", linestyle="--")
    ax.set_title(perfiles_ext.index[j]); ax.legend()
plt.show()

**¿Qué observas?** La recta se sale de [0, 1] en muchas muestras (en el perfil estable, en todas): el problema no es un caso raro, es estructural. La logística queda siempre dentro, y en el perfil de riesgo se acerca a 1 sin pasarse. Con la logística, el "techo" y el "suelo" están incorporados.

---
## 3. Probabilidad, odds y log-odds

`odds = p / (1 − p)` y `log-odds = ln(odds)`. La inversa del logit es la **sigmoide**: `p = 1 / (1 + e^(−η))`.

In [ ]:
def a_odds(p):     return p / (1 - p)
def a_logodds(p):  return np.log(p / (1 - p))
def sigmoide(eta): return 1 / (1 + np.exp(-eta))

ps = np.array([0.10, 0.20, 0.50, 0.80, 0.90])
tabla = pd.DataFrame({"p": ps, "odds": a_odds(ps), "log-odds": a_logodds(ps)})
display(tabla.round(3))
print("Comprobación: sigmoide(log-odds) devuelve p →", np.allclose(sigmoide(a_logodds(ps)), ps))

eta = np.linspace(-6, 6, 200)
plt.plot(eta, sigmoide(eta))
plt.axhline(0.5, color="gray", linestyle=":")
plt.title("La sigmoide: de log-odds a probabilidad")
plt.xlabel("log-odds (η)")
plt.ylabel("p")
plt.show()

### Leer un odds ratio: exp(0,69) ≈ 2

Un coeficiente β = 0,69 tiene un odds ratio de `exp(0,69) ≈ 2`: los **odds se duplican**. ¿Se duplica la probabilidad? Lo comprobamos partiendo de dos probabilidades distintas.

In [ ]:
beta = 0.69
or_ = np.exp(beta)
print("Odds ratio = exp(0,69) =", round(or_, 3))

for p0 in (0.20, 0.50):
    p1 = sigmoide(a_logodds(p0) + beta)          # sumar β a los log-odds = multiplicar los odds por el OR
    print(f"Parte de p = {p0:.2f}  (odds {a_odds(p0):.2f})  →  p = {p1:.3f}  (odds {a_odds(p1):.2f})   razón de probabilidades: {p1 / p0:.2f}")

**¿Qué observas?** Los odds se duplican siempre (de 0,25 a 0,50; de 1 a 2), pero la probabilidad pasa de 0,20 a 0,333 (×1,67) y de 0,50 a 0,667 (×1,33). El odds ratio **no es** el cociente de probabilidades (riesgo relativo).

---
## 4. Máxima verosimilitud en pequeño

Tres clientes con resultados (1, 0, 1) y probabilidades previstas (0,8; 0,3; 0,6). La verosimilitud es el producto de las probabilidades que el modelo asignaba a lo que **realmente ocurrió**.

In [ ]:
y3 = np.array([1, 0, 1])
p3 = np.array([0.8, 0.3, 0.6])
prob_observada = np.where(y3 == 1, p3, 1 - p3)
verosimilitud = prob_observada.prod()
print("Probabilidad de lo observado, cliente a cliente:", prob_observada)
print("Verosimilitud:", round(verosimilitud, 3), " Log-verosimilitud:", round(np.log(verosimilitud), 3))

---
## 5. Ajustar el modelo y leer la salida

In [ ]:
modelo = smf.logit("abandona ~ antiguedad + gasto + tickets + plan_anual", data=df).fit(disp=0)
print(modelo.summary().tables[1])     # tabla de coeficientes (la cabecera incluye fecha y hora, por eso la imprimimos aparte)
print("Observaciones:", int(modelo.nobs), " convergió:", modelo.mle_retvals["converged"])

Comparamos la estimación con los coeficientes **verdaderos** (que solo conocemos porque los datos son sintéticos) y comprobamos si cada uno cae dentro del intervalo de confianza al 95 %.

In [ ]:
ic = modelo.conf_int()
comparacion = pd.DataFrame({
    "verdadero": [VERDAD[v] for v in modelo.params.index],
    "estimado": modelo.params,
    "IC 2.5%": ic[0], "IC 97.5%": ic[1]})
comparacion["¿verdadero dentro del IC?"] = (comparacion["IC 2.5%"] <= comparacion["verdadero"]) & (comparacion["verdadero"] <= comparacion["IC 97.5%"])
display(comparacion.round(4))
print("Log-likelihood:", round(modelo.llf, 1), "  modelo nulo:", round(modelo.llnull, 1))
print("Pseudo R2:", round(modelo.prsquared, 3), "  AIC:", round(modelo.aic, 1))

### El mismo modelo con `sm.GLM` (familia Binomial)

Un GLM con familia Binomial y enlace logit (el predeterminado) es exactamente la regresión logística: los coeficientes coinciden. (El AIC aparece en la salida; lo usaremos para comparar modelos en la clase 12.)

In [ ]:
glm_bin = smf.glm("abandona ~ antiguedad + gasto + tickets + plan_anual", data=df, family=sm.families.Binomial()).fit()
print("Diferencia máxima entre coeficientes logit y GLM:", float(np.abs(glm_bin.params - modelo.params).max()) < 1e-8)

---
## 6. Odds ratio con intervalo de confianza

Aplicamos `exp` a los coeficientes **y a los extremos de su intervalo**. También reescalamos a un cambio relevante (12 meses, 10 €).

In [ ]:
tabla_or = pd.DataFrame({"odds ratio": np.exp(modelo.params),
                         "IC 2.5%": np.exp(ic[0]), "IC 97.5%": np.exp(ic[1])})
display(tabla_or.drop(index="Intercept").round(3))

print("12 meses más de antigüedad multiplican los odds por", round(np.exp(12 * modelo.params["antiguedad"]), 3))
print("10 € más de gasto multiplican los odds por", round(np.exp(10 * modelo.params["gasto"]), 3))

### Experimento guiado 2: odds ratio frente a probabilidad

El odds ratio de `tickets` (≈ 1,68) es el mismo para cualquier cliente. Lo aplicamos a probabilidades de partida muy distintas y miramos qué ocurre con la probabilidad. **Antes de ejecutar, predice:** ¿en qué punto de partida el odds ratio se parece más al cociente de probabilidades?

In [ ]:
or_tickets = float(np.exp(modelo.params["tickets"]))
p_partida = np.array([0.01, 0.05, 0.20, 0.50, 0.80, 0.95])
p_final = sigmoide(a_logodds(p_partida) + modelo.params["tickets"])
efecto = pd.DataFrame({"p de partida": p_partida, "p con +1 ticket": p_final,
                       "cociente de probabilidades": p_final / p_partida,
                       "diferencia (puntos)": (p_final - p_partida) * 100})
print("Odds ratio de tickets:", round(or_tickets, 3))
display(efecto.round(3))

plt.plot(p_partida * 100, (p_final - p_partida) * 100, marker="o")
plt.title("Mismo odds ratio, distinto cambio en probabilidad")
plt.xlabel("probabilidad de partida (%)")
plt.ylabel("cambio de p con +1 ticket (puntos)")
plt.show()

**¿Qué observas?** Con p = 0,01 el cociente de probabilidades es casi igual al odds ratio (≈ 1,67): suceso raro, odds ≈ probabilidad. Al subir la probabilidad de partida el cociente cae hacia 1 (con p = 0,95 apenas 1,02), y el cambio en puntos crece hasta un máximo cerca de p = 0,5 y vuelve a bajar. **El mismo odds ratio da cambios muy distintos**.

Lo mismo con datos reales: la tabla simple de los clientes con **3 o más tickets** frente al resto (sin modelo).

In [ ]:
df["muchos_tickets"] = (df["tickets"] >= 3).astype(int)
g = df.groupby("muchos_tickets")["abandona"].agg(["count", "sum", "mean"])
g.columns = ["clientes", "abandonan", "p"]
g["odds"] = g["p"] / (1 - g["p"])
display(g.round(3))
rr = g.loc[1, "p"] / g.loc[0, "p"]
odds_r = g.loc[1, "odds"] / g.loc[0, "odds"]
print("Riesgo relativo:", round(rr, 2), " Odds ratio:", round(odds_r, 2))

**¿Qué observas?** La probabilidad de abandonar se multiplica por ≈ 2,1 (riesgo relativo), pero los odds se multiplican por ≈ 3,0. Como el abandono es frecuente (24 %), el odds ratio **sobrestima** el cambio en probabilidad. Decir "el triple de probabilidad" sería un error.

---
## 7. Variables categóricas: dummies y categoría de referencia

Agrupamos la antigüedad en tres tramos y dejamos que `statsmodels` cree las variables dummy con `C(...)`. `Treatment(reference="nuevo")` fija la categoría de referencia.

In [ ]:
df["tramo"] = pd.cut(df["antiguedad"], [0, 11, 35, 72], labels=["nuevo", "medio", "veterano"]).astype(str)
display(df.groupby("tramo")["abandona"].agg(["count", "mean"]).round(3))

m_tramo = smf.logit("abandona ~ C(tramo, Treatment(reference='nuevo')) + gasto + tickets + plan_anual", data=df).fit(disp=0)
ic_t = np.exp(m_tramo.conf_int())
tabla_t = pd.DataFrame({"coef": m_tramo.params, "odds ratio": np.exp(m_tramo.params), "IC 2.5%": ic_t[0], "IC 97.5%": ic_t[1]})
display(tabla_t.filter(like="tramo", axis=0).round(3))

b_vet, b_med = [m_tramo.params[f"C(tramo, Treatment(reference='nuevo'))[T.{t}]"] for t in ("veterano", "medio")]
print("Odds ratio veterano frente a medio (no aparece en la tabla):", round(np.exp(b_vet - b_med), 3))

**¿Qué observas?** Cada odds ratio compara con la referencia `nuevo`: el `veterano` tiene unos odds de abandono ≈ 75 % menores (odds ratio ≈ 0,25) y el `medio` ≈ 33 % menores. Para comparar `veterano` con `medio` hay que dividir los dos odds ratio (≈ 0,38). Si cambiáramos la referencia, los coeficientes cambiarían de valor: por eso hay que decir siempre "frente a qué".

---
## 8. Probabilidades previstas y efecto marginal medio

El odds ratio de `tickets` es el mismo para todos los clientes, pero **el cambio en probabilidad depende del punto de partida**. Comparamos dos perfiles y luego promediamos sobre todos los clientes (efecto marginal medio).

In [ ]:
perfiles = pd.DataFrame({
    "antiguedad": [48, 48, 6, 6], "gasto": [30, 30, 50, 50],
    "tickets": [0, 1, 3, 4], "plan_anual": [1, 1, 0, 0]},
    index=["tranquilo, 0 tickets", "tranquilo, 1 ticket",
           "en riesgo, 3 tickets", "en riesgo, 4 tickets"])
perfiles["p"] = modelo.predict(perfiles)
display(perfiles.round(3))
print("Cambio de p con +1 ticket (tranquilo):", round((perfiles["p"].iloc[1] - perfiles["p"].iloc[0]) * 100, 1), "puntos porcentuales")
print("Cambio de p con +1 ticket (en riesgo):", round((perfiles["p"].iloc[3] - perfiles["p"].iloc[2]) * 100, 1), "puntos porcentuales")

In [ ]:
t = np.arange(0, 9)
for nombre, a, g_, pl in [("tranquilo", 48, 30, 1), ("en riesgo", 6, 50, 0)]:
    datos_t = pd.DataFrame({"antiguedad": a, "gasto": g_, "tickets": t, "plan_anual": pl})
    plt.plot(t, modelo.predict(datos_t), marker="o", label=nombre)
plt.title("Probabilidad de abandono según los tickets")
plt.xlabel("tickets")
plt.ylabel("probabilidad prevista")
plt.legend()
plt.show()

meses = np.arange(1, 73)
for pl, estilo in [(0, "-"), (1, "--")]:
    datos_m = pd.DataFrame({"antiguedad": meses, "gasto": 40, "tickets": 1, "plan_anual": pl})
    plt.plot(meses, modelo.predict(datos_m), estilo, label=f"plan_anual = {pl}")
plt.title("Probabilidad de abandono según la antigüedad y el plan")
plt.xlabel("antigüedad (meses)")
plt.ylabel("probabilidad prevista")
plt.legend()
plt.show()

**¿Qué observas?** Un ticket adicional mueve la probabilidad unos 2 puntos en el cliente tranquilo y unos 8 en el de riesgo, con **el mismo odds ratio**. Las curvas son S, no rectas. En el segundo gráfico, el plan anual tiene siempre el mismo odds ratio (≈ 0,47), pero la distancia entre las dos curvas es grande con poca antigüedad (probabilidades altas) y casi desaparece con mucha (probabilidades bajas).

### Experimento guiado 3: el efecto marginal de un ticket, cliente a cliente

Calculamos el cambio de probabilidad al añadir un ticket **para cada uno de los 2 000 clientes** y miramos su distribución. Su media es el efecto marginal medio (AME). **Antes de ejecutar, predice:** ¿el efecto será parecido para todos los clientes? ¿Entre qué valores esperas que se mueva?

In [ ]:
p_antes = modelo.predict(df)
df_mas = df.copy()
df_mas["tickets"] = df_mas["tickets"] + 1
efecto_ind = (modelo.predict(df_mas) - p_antes) * 100
print("Efecto marginal medio (AME) de +1 ticket:", round(efecto_ind.mean(), 1), "puntos porcentuales")
print("Efecto individual mínimo y máximo:", round(efecto_ind.min(), 1), "y", round(efecto_ind.max(), 1))
me = modelo.get_margeff()
print("Versión con derivadas (get_margeff):", round(me.margeff[list(modelo.params.index[1:]).index("tickets")] * 100, 1))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
a1.hist(efecto_ind, bins=30, edgecolor="white")
a1.axvline(efecto_ind.mean(), color="red", linestyle="--", label="AME")
a1.set_title("Efecto de +1 ticket, cliente a cliente")
a1.set_xlabel("cambio de p (puntos)"); a1.legend()
a2.scatter(p_antes * 100, efecto_ind, s=6, alpha=0.4)
a2.set_title("Efecto según la probabilidad de partida")
a2.set_xlabel("p de partida (%)"); a2.set_ylabel("cambio de p (puntos)")
plt.show()

**¿Qué observas?** El efecto **no es el mismo para todos**: va de ≈ 1,6 a ≈ 12,9 puntos, y el AME (≈ 9,3) es un resumen. El diagrama de dispersión muestra la relación con la probabilidad de partida: el efecto es pequeño en los extremos y máximo en la zona media. Un único "efecto de un ticket" en puntos solo tiene sentido como promedio.

---
## 9. Separación perfecta (solo un aviso)

Si una variable separa por completo los 0 de los 1, la verosimilitud no tiene máximo finito y los coeficientes se disparan. Según la versión de `statsmodels`, verás un aviso o un error de matriz singular. **No se interpreta.**

In [ ]:
d_sep = pd.DataFrame({"x": [1, 2, 3, 4, 5, 6, 7, 8], "y": [0, 0, 0, 0, 1, 1, 1, 1]})   # x separa perfectamente y
hubo_problema = False
with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    try:
        smf.logit("y ~ x", data=d_sep).fit(disp=0)
    except Exception:
        hubo_problema = True
hubo_problema = hubo_problema or len(avisos) > 0
print("¿statsmodels avisó o falló ante la separación perfecta?", hubo_problema)

---
## 10. La familia GLM y la regresión de Poisson (solo como idea)

Un GLM tiene una **componente aleatoria**, un **predictor lineal** y una **función de enlace**. Solo cambia la familia en `smf.glm`.

In [ ]:
familias = pd.DataFrame({
    "componente aleatoria": ["Normal (Gaussian)", "Binomial", "Poisson"],
    "enlace": ["identidad", "logit", "logaritmo"],
    "respuesta típica": ["continua (precio)", "0/1 (abandona)", "conteo (tickets)"]},
    index=["lineal", "logística", "Poisson"])
display(familias)

# Poisson: el número de tickets es un conteo. exp(coef) multiplica la TASA (conteo medio), no los odds
poisson = smf.glm("tickets ~ gasto", data=df, family=sm.families.Poisson()).fit()
print("Coeficiente de gasto (log de la tasa por €):", round(poisson.params["gasto"], 4))
print("10 € más de gasto multiplican el nº medio de tickets por", round(np.exp(10 * poisson.params["gasto"]), 3))

**¿Qué observas?** El código es el mismo que en la logística: solo cambia `family`. El coeficiente se interpreta en la escala del enlace (aquí, log de la tasa) y `exp(β)` multiplica la **tasa media de tickets**, no unos odds. Poisson aquí es solo un ejemplo de por qué existen los GLM: para conteos la lineal y la logística no sirven.

---
## Resumen de la clase

- La regresión lineal no sirve para un 0/1; el modelo logístico modela los **log-odds** como función lineal y devuelve probabilidades en (0, 1).
- Los coeficientes son **log-odds**; `exp(β)` es el **odds ratio** (no la probabilidad ni el riesgo relativo). Se informa con su intervalo.
- Con variables categóricas, cada odds ratio compara con la **categoría de referencia**.
- El cambio en probabilidad depende del punto de partida; el **efecto marginal medio** lo resume.
- Una separación perfecta produce coeficientes que no se interpretan.
- Los **GLM** unifican lineal, logística y Poisson: componente aleatoria + predictor lineal + enlace.

**Siguiente paso:** el notebook de práctica de esta clase y después la clase 12: Multicolinealidad, selección de variables y validación.